Purpose: Find the common & unique DEGs across species & physiologies.<br>
Author: Anna Pardo<br>
Date initiated: Apr. 30, 2026

In [14]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import venn
from collections import Counter

In [3]:
# set directory paths
local = "./DE_results/"
hpc = "./DEGs_results/"

In [6]:
def get_degs_file(filepath):
    df = pd.read_csv(filepath,delim_whitespace=True,quotechar='"')
    genes = list(df.index)
    return genes

In [7]:
# load parental DEGs - Yf
yfde = get_degs_file(os.path.join(local,"degs_Yf.txt"))

In [9]:
# get Ya DEGs
yade = get_degs_file(os.path.join(local,"degs_Ya.txt"))

In [95]:
with open("./Ya_DEGs_list.txt","w+") as outfile:
    for i in yade:
        outfile.write(i+"\n")

In [96]:
with open("./Yf_DEGs_list.txt","w+") as outfile:
    for i in yfde:
        outfile.write(i+"\n")

In [10]:
# get Yg DEGs run locally (three subsets of C3+CAM Yg)
localruns = {}
for f in os.listdir(local):
    if (not f.endswith("_pval.txt")) and (not f.endswith("_R2.txt")):
        if ("Yf" not in f) and ("Ya" not in f):
            ID = f.split("s_")[1].split(".")[0]
            localruns[ID] = get_degs_file(os.path.join(local,f))

In [11]:
# get the rest of the information
hpcruns = {}
for f in os.listdir(hpc):
    if f.endswith("genes.txt"):
        ID = f.split("_3reps")[0]
        hpcruns[ID] = get_degs_file(os.path.join(hpc,f))

In [12]:
localruns.update(hpcruns)

In [13]:
# localruns now contains all the DEG info
## split into three dicts: one for each physiology
camd = {k:v for k,v in localruns.items() if ("facultative" not in k) and ("C3" not in k)}
fcd = {k:v for k,v in localruns.items() if "facultative" in k}
c3d = {k:v for k,v in localruns.items() if "C3" in k}

In [22]:
def genes_in_at_least_x_entries(d, max_x=10):
    """
    d: dictionary of lists {key: [genes, ...]}
    max_x: highest X to report
    
    Returns:
        counts_by_x -> {X: number of genes found in at least X entries}
        gene_freqs   -> Counter of how many entries each gene appears in
    """
    
    # Count how many dictionary entries each gene appears in
    gene_freqs = Counter()
    
    for gene_list in d.values():
        for gene in set(gene_list):   # avoids duplicates within one list
            gene_freqs[gene] += 1
            
    # set up a dataframe of number of sets in which each DEG is found
    gene_freqs = dict(gene_freqs)
    gfd = {"GeneID":list(gene_freqs.keys()),"nsets":list(gene_freqs.values())}

    return pd.DataFrame(gfd)

In [23]:
camfreqs = genes_in_at_least_x_entries(camd)
c3freqs = genes_in_at_least_x_entries(c3d)
facfreqs = genes_in_at_least_x_entries(fcd)

In [25]:
camfreqs["phys"] = "CAM"
camfreqs["species"] = "Yg"

In [26]:
c3freqs["phys"] = "C3+CAM"
c3freqs["species"] = "Yg"

In [27]:
facfreqs["phys"] = "facultative_CAM"
facfreqs["species"] = "Yg"

In [28]:
# stick these together
ygfreqs = pd.concat([camfreqs,c3freqs,facfreqs])

In [29]:
# subset to only DEGs that were DE in >= 9 data subsets
yg9 = ygfreqs[ygfreqs["nsets"]>=9]

In [31]:
len(yg9["GeneID"].unique())

69

In [94]:
yg9.to_csv("./Yg_maSigPro_DEGs_allphys.txt",sep="\t",header=True,index=False)

In [35]:
# pull out Yg DEG lists
camde = set(list(yg9[yg9["phys"]=="CAM"]["GeneID"]))
c3de = set(list(yg9[yg9["phys"]=="C3+CAM"]["GeneID"]))
facde = set(list(yg9[yg9["phys"]=="facultative_CAM"]["GeneID"]))

In [34]:
set.intersection(*[set(camde),set(c3de),set(facde)])

set()

In [48]:
cam_c3cam = list(camde.intersection(c3de))

In [37]:
camde.intersection(facde)

set()

In [38]:
facde.intersection(c3de)

set()

In [47]:
ya_cam = list(set.intersection(*[set(yade),camde]))

In [46]:
ya_c3cam = list(set.intersection(*[set(yade),c3de]))

In [41]:
set.intersection(*[set(yade),facde])

set()

In [42]:
set.intersection(*[set(yfde),facde])

set()

In [45]:
yf_cam = list(set.intersection(*[set(yfde),camde]))

In [44]:
set.intersection(*[set(yfde),c3de])

set()

In [65]:
# what are the functions of these genes?
## load files from genome annotations of parents
yfgf = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/gene.functions.csv",sep=",",header="infer",
                  comment="#")
yfgf.head()

,GeneID,Id,IdType,"description if any, GO is via pfam2go/interpro"
0,YufilH1039891m.g,K01213,KEGGORTH,"galacturan 1,4-alpha-galacturonidase [EC:3.2.1..."
1,YufilH1039891m.g,GO:0005975,GO,carbohydrate metabolic process
2,YufilH1039891m.g,PF00295,PFAM,Glycosyl hydrolases family 28
3,YufilH1039891m.g,3.2.1.15,EC,Polygalacturonase.
4,YufilH1039891m.g,GO:0004650,GO,polygalacturonase activity


In [66]:
yf_cam_func = yfgf[yfgf["GeneID"].isin(yf_cam)]

In [64]:
yagf = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/Y_aloifolia.emapper.annotations.csv",sep=",",
                  header="infer",comment="#")
yagf.head()

,query,seed_ortholog,evalue,score,eggNOG_OGs,max_annot_lvl,COG_category,Description,Preferred_name,GOs,...,KEGG_ko,KEGG_Pathway,KEGG_Module,KEGG_Reaction,KEGG_rclass,BRITE,KEGG_TC,CAZy,BiGG_Reaction,PFAMs
0,Yucal.1Z013500.1.p,4641.GSMUA_Achr1P08280_001,2.290000e-161,467.0,"28JMV@1|root,2QS10@2759|Eukaryota,37J0Q@33090|...",35493|Streptophyta,S,ACT domain containing protein,-,-,...,-,-,-,-,-,-,-,-,-,"ACT,ACT_6"
1,Yucal.1Z065500.1.p,42345.XP_008781230.1,4.650000e-135,397.0,"COG0515@1|root,KOG0192@2759|Eukaryota,37N3E@33...",35493|Streptophyta,T,Serine threonine-protein kinase,-,"GO:0003674,GO:0003824,GO:0004672,GO:0005575,GO...",...,ko:K00799,"ko00480,ko00980,ko00982,ko00983,ko01524,ko0520...",-,"R03522,R07002,R07003,R07004,R07023,R07024,R070...","RC00004,RC00069,RC00840,RC00948,RC01704,RC0170...","ko00000,ko00001,ko01000,ko02000","1.A.12.2.2,1.A.12.3.2",-,-,"ACT,Pkinase_Tyr"
2,Yucal.1Z302100.1.p,42345.XP_008785794.1,2.360000e-150,440.0,"KOG3683@1|root,KOG3683@2759|Eukaryota,37M6K@33...",35493|Streptophyta,S,DUF1767,-,-,...,ko:K18404,-,-,-,-,"ko00000,ko03019,ko03036",-,-,-,RMI1_N
3,Yucal.1Z302200.1.p,42345.XP_008809006.1,6.420000e-124,360.0,"COG0605@1|root,KOG0876@2759|Eukaryota,37RMW@33...",35493|Streptophyta,P,radicals which are normally produced within th...,-,"GO:0000302,GO:0000303,GO:0000305,GO:0003674,GO...",...,ko:K04564,"ko04013,ko04068,ko04146,ko04211,ko04212,ko0421...",-,-,-,"ko00000,ko00001,ko01000",-,-,-,"Sod_Fe_C,Sod_Fe_N"
4,Yucal.1Z302000.1.p,42345.XP_008785831.1,4.730000e-264,749.0,"28JRK@1|root,2QS51@2759|Eukaryota,37MB5@33090|...",35493|Streptophyta,S,Protein FAR1-RELATED SEQUENCE,-,"GO:0003674,GO:0003700,GO:0005575,GO:0005622,GO...",...,ko:K17604,-,-,-,-,"ko00000,ko01009",-,-,-,"FAR1,MULE,SWIM"


In [67]:
yagf = yagf[["query","Description"]].rename(columns={"query":"GeneID"})
yagf.head()

,GeneID,Description
0,Yucal.1Z013500.1.p,ACT domain containing protein
1,Yucal.1Z065500.1.p,Serine threonine-protein kinase
2,Yucal.1Z302100.1.p,DUF1767
3,Yucal.1Z302200.1.p,radicals which are normally produced within th...
4,Yucal.1Z302000.1.p,Protein FAR1-RELATED SEQUENCE


In [68]:
ya_cam

['Yucal.06G098700.v2.1', 'Yucal.05G264600.v2.1']

In [69]:
yagf["GeneID"] = yagf["GeneID"].str.rstrip(".1.p")
yagf.head()

,GeneID,Description
0,Yucal.1Z013500,ACT domain containing protein
1,Yucal.1Z065500,Serine threonine-protein kinase
2,Yucal.1Z302100,DUF1767
3,Yucal.1Z302200,radicals which are normally produced within th...
4,Yucal.1Z302000,Protein FAR1-RELATED SEQUENCE


In [70]:
yagf["GeneID"] = yagf["GeneID"]+".v2.1"
yagf.head()

,GeneID,Description
0,Yucal.1Z013500.v2.1,ACT domain containing protein
1,Yucal.1Z065500.v2.1,Serine threonine-protein kinase
2,Yucal.1Z302100.v2.1,DUF1767
3,Yucal.1Z302200.v2.1,radicals which are normally produced within th...
4,Yucal.1Z302000.v2.1,Protein FAR1-RELATED SEQUENCE


In [71]:
yacam_func = yagf[yagf["GeneID"].isin(ya_cam)]
yac3_func = yagf[yagf["GeneID"].isin(ya_c3cam)]
camc3_func = yagf[yagf["GeneID"].isin(cam_c3cam)]

In [72]:
yac3_func

,GeneID,Description
14466,Yucal.06G001500.v2.1,JAB/MPN domain
16997,Yucal.05G188100.v2.1,Belongs to the carbohydrate kinase PfkB family
31063,Yucal.27G027800.v2.1,Cupin-like domain


In [73]:
yacam_func

,GeneID,Description
13658,Yucal.06G098700.v2.1,Vacuolar-processing enzyme
16736,Yucal.05G264600.v2.1,EID1-like F-box protein 2


In [74]:
camc3_func

,GeneID,Description
11430,Yucal.17G042300.v2.1,DnaJ domain
27375,Yucal.04G281300.v2.1,Alcohol dehydrogenase GroES-like domain


In [79]:
yfcam_func = yf_cam_func[["GeneID","description if any, GO is via pfam2go/interpro"]].rename(columns={
    "description if any, GO is via pfam2go/interpro":"Description"
})
yfcam_func.dropna(axis=0,inplace=True)

In [82]:
yfcam_func.drop_duplicates(inplace=True)

In [85]:
yfcam_func["GeneID"].unique()

array(['YufilH1044172m.g', 'YufilH1059138m.g', 'YufilH1061746m.g',
       'YufilH1082582m.g', 'YufilH1024614m.g', 'YufilH1064049m.g'],
      dtype=object)

In [89]:
yfcam_func[yfcam_func["GeneID"]=="YufilH1064049m.g"]["Description"].unique()

array(['glutamine biosynthetic process',
       'Glutamine synthetase, catalytic domain', 'Glutamine synthetase',
       'glutamate-ammonia ligase activity',
       'glutamine synthetase [EC:6.3.1.2]',
       'nitrogen compound metabolic process', 'GLUTAMINE SYNTHETASE',
       'Glutamate--ammonia ligase.',
       'Glutamine synthetase, beta-Grasp domain'], dtype=object)